# Lab 6: What can a sample tell us about an average?

Use fictional study data to estimate one population mean, show its precision, and check when the interval method works. The lab opens with the neighborhood join because it determines which people and context records can be interpreted.

**Overall question:** How can one sample estimate an average for a larger group, and how much could that estimate vary?

**Learning objectives**
- Join the people and neighborhood tables without changing the person-level population or turning missing context into a score.
- Distinguish individual variation from variation among sample means, and explain how sample size affects the standard error.
- Construct a confidence interval for a population mean and check its coverage across repeated samples.

There are three Your Turns. Canvas labels identify the ten reflection questions; complete the requested assessment in Canvas, not in this notebook. The worked ACE10 example is shared by everyone; your selected ACE10 or cohesion analysis is a separate workflow. Seeds, sample sizes and repetitions are fixed so the supplied reference values remain reproducible.


## Section 1. Measures and plan

Each row represents one person at the outcome visit used in this lab. Choose ACE10 or cohesion before viewing results; the common demonstration uses ACE10, and your later sample is separate.

| Measure | What it tells us | How to read it |
| --- | --- | --- |
| **ACE10** | Categories of childhood adversity reported by a fictional young person. | Count from 0 to 10; higher means more categories, not more events. |
| **Family cohesion** | Perceived commitment, help and support in a young person's family. | Score from 0 to 1; higher indicates greater perceived cohesion. |
| **Broad cannabis-use days** | Distinct days of any of eight products in the 30 completed days before assessment. | Count from 0 to 30; a day counts once even if several products were used. |
| **COI** | Neighborhood conditions and resources summarized from several sources. | Higher means greater opportunity; compare Very low through Very high. |

Cohesion is a perception of family support, not the opposite of ACE10; 0.75 is a scale score, not 75% of families. A zero is a reported value; missing means unavailable. COI describes a neighborhood, not an individual.

Set `CHOICE = 'adversity'` for ACE10 or `'cohesion'` for cohesion. The ACE10 teaching count has an age-capped window and is not a clinical instrument or official released ABCD score. Your choice defines the outcome and units used in the later mean estimate and interval. Column definitions, codes and missing-value rules are in the [data dictionary](../data/labs_06_07/DATA_DICTIONARY.md).


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

VERSION = '0.6.0-rc1'
NOTEBOOK_VERSION = '3.0.0'
DATA = Path('../data/labs_06_07')
assert (DATA/'instructional_analysis.csv').is_file(), 'Open this notebook from the labs/ folder of the course repository.'
from l6_opening_support import show, seed_rng, summarize_discovery
from l6_join_checks import check_context_lookup, check_join_keys, check_release_eligibility

TABLES = DATA
ACE = 'instructional_ace10'
DAYS = 'instructional_cannabis_days_30d'
COI_SCORE = 'le_l_coi__addr1__coi__total__national_score'
COI_LEVEL = 'le_l_coi__addr1__coi__total__national_c5'
KEYS = ['participant_id','session_id']
columns = KEYS + [ACE,'instructional_ace_status','fc_y_fes__cohes_mean',
    'instructional_cohesion_status', DAYS,'instructional_cannabis_status',
    'su_y_tlfb__mj__smoke__1mo_ud','instructional_smoking_status']

# Start without prejoined COI; reconstruct it from the supplied source/link tables.
outcomes = pd.read_csv(TABLES/'instructional_analysis.csv', usecols=columns).sort_values(KEYS).reset_index(drop=True)
links = pd.read_csv(TABLES/'instructional_baseline_link.csv')
context = pd.read_csv(TABLES/'instructional_context.csv')
coi_reference = pd.read_csv(TABLES/'le_l_coi.csv')
assert len(outcomes) == 8564 and outcomes['session_id'].eq('ses-07A').all() and outcomes['participant_id'].is_unique
assert links['participant_id'].is_unique and links['context_event'].eq('ses-00A').all()
check_context_lookup(context)
assert context['instructional_coi_vintage'].eq(2015).all()
pd.set_option('display.precision', 3)
print('Verified student data:', VERSION, '| fictional outcome records:', len(outcomes))



In [ ]:
# Your independent outcome choice; make it before viewing the exploration results.

CHOICE = 'adversity'  # 'adversity' (ACE10) or 'cohesion'
MENU = {
    'adversity': {'field': ACE, 'status':'instructional_ace_status',
                 'units':'ACE categories per person (0–10)', 'role':1},
    'cohesion': {'field':'fc_y_fes__cohes_mean', 'status':'instructional_cohesion_status',
                'units':'cohesion score units (0–1)', 'role':2}}
assert CHOICE in MENU, 'Choose adversity or cohesion. Cannabis is not an independent inference choice.'
meta = MENU[CHOICE]
field, status = meta['field'], meta['status']
analysis_plan = {
    'dataset_version':VERSION, 'realization':0, 'notebook_version':NOTEBOOK_VERSION,
    'choice':CHOICE, 'field':field, 'units':meta['units'], 'outcome_visit':'ses-07A',
    'population':'all usable values of the selected outcome in the fixed fictional roster',
    'unit':'one participant at the selected outcome visit', 'COI_required_for_inference':False,
    'inference_n':[75,300], 'confidence':0.95, 'confidence_comparison':[0.90,0.95,0.99],
    'workflow_revision':'narrative-turn3-2026-11-10', 'method':'normal reference; sample SD/sqrt(n); no FPC',
    'sampling':'SRS without replacement within a sample; restore roster between draws',
    'classroom_repetitions':2000, 'seed_root':6060, 'role_id':meta['role'],
    'seed_fields':['root','role','operation','n','run'],
    'exploration_n':1000, 'exploration_streams':[[6060,4,3,1000,1],[6060,4,3,1000,2]]}
source_record = pd.DataFrame([
    {'measure':'ACE10','reference':'fictional youth report; age-capped childhood window',
     'limits':'13 responses → 10 categories; categories are not broad domains or raw codes'},
    {'measure':'Cohesion','reference':'fictional general family perception at assessment',
     'limits':'at least 8/9 valid direction-coded items; no invented 30-day recall'},
    {'measure':'Broad cannabis','reference':'distinct days in 30 completed days; union across 8 products',
     'limits':'use for discovery; keep documented nonuse zeros; unavailable reports stay missing'},
    {'measure':'Smoked flower','reference':'narrower product-specific distinct days in the same window',
     'limits':'later conditions contrast, not another independent mean-analysis choice'},
    {'measure':'COI','reference':'historical baseline; COI 2.0; national; 2015; 2010 geography',
     'limits':'fictional area score; unchanged residence is a simulation assumption, not measured history'}])
print(f"Plan recorded: {CHOICE} | {meta['units']} | n=75/300 | two-sided 95% interval")
# Keep the complete analysis_plan and source_record for the final record.
source_card = pd.DataFrame([
    {'measure':'ACE10', 'meaning':'13 fictional youth-report responses → 10 categories; count 0–10',
     'limit':'Window ends at assessment or age 18, whichever comes first; younger respondents have incomplete childhood observation. Not a released ABCD score or clinical instrument.'},
    {'measure':'Cohesion', 'meaning':'General family perception; 0–1 mean of ≥8/9 valid, direction-coded items',
     'limit':'Distinct from ACE10; no invented 30-day recall.'},
    {'measure':'Broad cannabis', 'meaning':'Distinct use-days in 30 completed days; union across eight products',
     'limit':'Documented nonuse is zero; unavailable reports remain missing.'},
    {'measure':'COI', 'meaning':'Baseline historical context: COI 2.0, national, 2015, 2010 geography',
     'limit':'Fictional area score; unchanged residence is a simulation assumption, not measured history.'}])
show(source_card)


### What to notice

The target is the mean among people in the fictional roster with a usable value for the selected measure, not all adolescents in the United States. The plan and outcome units are fixed before results; next, the join adds historical neighborhood context without changing who can contribute a score.


## Section 2. Join and eligibility

The outcome table has one row per person at `ses-07A`; a link table gives each person's area at baseline visit `ses-00A`; a context table supplies that area's COI. Keys identify the records to match: participant ID links a person to the baseline area, then area plus source year identifies the context row. Matching area alone can return several years.

The lookup uses 2015 COI scores and 2010 geographic boundaries as historical context. The fictional study assumes residence stayed the same; this does not establish a current neighborhood or a person's resources. Keep an unmatched COI value missing; it is not zero or low opportunity and does not by itself make a usable outcome ineligible.

### Demo: why keys and relationship checks matter

In the second demo cell, each person-visit has one record in each table, so `validate='one_to_one'` checks that neither side repeats the join key. Several people may share one area-year context record, so the context join uses `validate='many_to_one'`: many rows on the left may match one unique lookup row on the right. Validation checks the expected relationship and rejects accidental duplication.

The demo compares an area-only join with area plus year. Check row counts, uniqueness, unmatched context and the requested vintage before interpreting the result. The expected relationship is determined by the two tables' keys, not by whether the returned values look plausible.

In [ ]:
toy_links = pd.DataFrame({'person':['Example 1','Example 2','Example 3'],
                         'area':['A','A','B'], 'vintage':[2015,2010,2015]})
toy_context = pd.DataFrame({'area':['A','A','B','B'], 'vintage':[2010,2015,2010,2015],
                           'score':[30,45,65,70]})
show(toy_links); show(toy_context)


In [ ]:
toy_wrong = toy_links.merge(toy_context, on='area', how='left')
try:
    toy_links.merge(toy_context, on='area', how='left', validate='many_to_one')
except pd.errors.MergeError:
    print('The many-to-one safeguard rejects area-only matching: an area has two lookup rows.')
toy_correct = toy_links.merge(toy_context, on=['area','vintage'], how='left', validate='many_to_one')
# One person's rows make the duplication visible without printing both full joins.
toy_comparison = pd.concat([
    toy_wrong.loc[toy_wrong['person'].eq('Example 1')]
        .rename(columns={'vintage_x':'requested_vintage','vintage_y':'matched_vintage'})
        .assign(join='area only'),
    toy_correct.loc[toy_correct['person'].eq('Example 1')]
        .assign(requested_vintage=lambda d:d['vintage'], matched_vintage=lambda d:d['vintage'],
                join='area + vintage')
], ignore_index=True)[['join','person','area','requested_vintage','matched_vintage','score']]
show(toy_comparison)
print(f'All people: {len(toy_links)} rows before; {len(toy_wrong)} area-only; {len(toy_correct)} with area and vintage.')


In [ ]:
# Person-visit records: each table has one row per person per visit, so the join is one-to-one.
visit_scores = pd.DataFrame({'participant_id':['P01','P01','P02'], 'session_id':['visit_A','visit_B','visit_A'],
                             'score':[2,3,0]})
visit_context = pd.DataFrame({'participant_id':['P01','P01','P02'], 'session_id':['visit_A','visit_B','visit_A'],
                              'context':['Low','High','Moderate']})
visit_join = visit_scores.merge(visit_context, on=['participant_id','session_id'], how='left', validate='one_to_one')
show(visit_join)
try:
    visit_scores.merge(pd.concat([visit_context, visit_context.iloc[[0]]]), on=['participant_id','session_id'],
                       how='left', validate='one_to_one')
except pd.errors.MergeError:
    print('one_to_one rejects a repeated person-visit record. Matching on participant_id alone would pair each of P01\'s two visits with both context rows.')


### Your Turn 1: join context and keep eligible people

Start with one row per person at the outcome visit. Work through the three placeholders in order: identify the lookup keys, choose the join direction that preserves the outcome rows, then combine the scored-status and nonmissing-value checks row by row with `&`. A Boolean Series has one True/False value per row; unlike `and`, `&` combines aligned row-wise checks.

Before running the cell, use these cases as mental checks: a scored and present value (including zero) can contribute; a present number with an unscored status cannot; a missing outcome value cannot; and an unmatched COI value stays missing without excluding an otherwise eligible outcome. Check that the joined table keeps one row per participant/visit and preserves the starting row count. No written notebook response is required.

**Further learning with Copilot (optional)**

> Use a tiny invented example to predict which rows stay in a left join and what `validate` checks. Then quiz me on a scored zero, a missing outcome, and an unmatched context record. Give one hint at a time; do not supply my settings or code.


In [ ]:
link_work = outcomes[KEYS].merge(
    links[['participant_id','instructional_area_id']], on='participant_id',
    how='left', validate='one_to_one').assign(instructional_coi_vintage=2015)

join_keys = None       # List of geographic lookup columns
join_how = None        # Join direction that retains every participant
status_ok = outcomes[status].eq('scored')
value_present = outcomes[field].notna()
eligible_mask = None   # TODO: combine both checks with `&`

if join_keys is None or join_how is None or eligible_mask is None:
    raise ValueError('Complete the three Your Turn 1 placeholders, then rerun this cell.')
    
check_join_keys(join_keys, link_work, context)
linked = link_work.merge(context[join_keys + [COI_SCORE,COI_LEVEL]],
    on=join_keys, how=join_how, validate='many_to_one', indicator=True)
assert len(linked) == len(outcomes), 'The join changed the row count. Check its keys and direction.'
assert not linked.duplicated(KEYS).any(), 'The join created more than one row per participant/visit.'
assert linked['participant_id'].tolist() == outcomes['participant_id'].tolist(), 'Preserve outcome row order.'
roster = outcomes.merge(linked[KEYS + [COI_SCORE,COI_LEVEL]], on=KEYS,
    how='left', validate='one_to_one')
assert isinstance(eligible_mask, pd.Series) and eligible_mask.index.equals(outcomes.index), 'Use an aligned Boolean Series.'
assert pd.api.types.is_bool_dtype(eligible_mask.dtype) and not eligible_mask.isna().any(), 'Eligibility must be True/False for every row.'
eligible = roster.loc[eligible_mask].copy()
check_release_eligibility(DATA, CHOICE, eligible['participant_id'])
assert eligible['participant_id'].is_unique and len(eligible) >= 300
assert 300/len(eligible) <= .05, 'The larger inference sample exceeds its registered fraction cap.'


In [ ]:
# Supplied reconstruction check: compare to the documented historical derived view.
assert coi_reference['session_id'].eq('ses-00A').all()
assert coi_reference['participant_id'].is_unique
coi_check = linked[['participant_id',COI_SCORE,COI_LEVEL]].merge(
    coi_reference[['participant_id',COI_SCORE,COI_LEVEL]], on='participant_id',
    how='outer', validate='one_to_one', suffixes=('_joined','_supplied'), indicator='check_source')
assert coi_check['check_source'].eq('both').all(), 'Participant sets differ from the supplied baseline view.'
for column in [COI_SCORE,COI_LEVEL]:
    a,b = coi_check[column+'_joined'],coi_check[column+'_supplied']
    assert (a.eq(b) | (a.isna() & b.isna())).all(), 'Reconstructed COI disagrees with its supplied derived view.'
linkage_audit = pd.DataFrame([{'outcome_rows':len(outcomes),'joined_rows':len(linked),
    'matched_context':int(linked['_merge'].eq('both').sum()),
    'unmatched_context':int(linked['_merge'].eq('left_only').sum()),
    'unique_participant_visit':not linked.duplicated(KEYS).any(), 'derived_view_agreement':True}])
population_audit = pd.DataFrame([{'outcome':field,'roster_N':len(roster),'eligible_N':len(eligible),
    'unavailable':len(roster)-len(eligible),'eligible_zeros':int(eligible[field].eq(0).sum()),
    'eligible_without_COI':int(eligible[COI_SCORE].isna().sum()),
    'fraction_n75':75/len(eligible),'fraction_n300':300/len(eligible)}])
# Preserve both full audit objects; display one short denominator trail.
audit_checkpoint = pd.DataFrame([
    {'checkpoint':'Outcome rows / joined rows', 'value':f"{len(outcomes):,} / {len(linked):,}"},
    {'checkpoint':'Matched / unmatched context', 'value':f"{int(linkage_audit.iloc[0].matched_context):,} / {int(linkage_audit.iloc[0].unmatched_context):,}"},
    {'checkpoint':'Eligible selected outcome / unavailable', 'value':f"{len(eligible):,} / {len(roster)-len(eligible):,}"},
    {'checkpoint':'Eligible zeros / eligible without COI', 'value':f"{int(population_audit.iloc[0].eligible_zeros):,} / {int(population_audit.iloc[0].eligible_without_COI):,}"}])
show(audit_checkpoint)
print('Checks passed: participant/visit uniqueness; agreement with the supplied historical COI view.')


### What to notice

The join preserves the outcome rows and uniqueness. Missing COI is neither zero nor an eligibility rule; agreement with `le_l_coi` checks the reconstruction, not the accuracy of historical information. Use `analysis_plan`, `linkage_audit` and `population_audit` for **L6-01–02**.


## Section 3. Why sampling matters

Before estimating a mean, the common ACE10 demonstration describes any reported cannabis use across ACE10 bands in Sample A. It includes 1,000 people with usable ACE10 scores; each band's denominator is people with a nonmissing broad-use report, including reported zeros. Missing reports are not counted as nonuse. COI availability does not determine the ACE10 sample.

The supplied summary table reports users, usable reports and the any-use percentage for each band. Read the denominator before comparing percentages. This is a description of Sample A, not a population or causal conclusion.


In [ ]:
# Provided sampling and band definitions; the next Your Turn creates Sample A counts.
demo_ace_roster = roster.loc[roster['instructional_ace_status'].eq('scored') & roster[ACE].notna()].sort_values(KEYS).reset_index(drop=True)
exploration_samples = {}
for label, run in [('A',1),('B',2)]:
    rng = seed_rng(role=4, operation=3, n=1000, run=run)
    indices = rng.choice(len(demo_ace_roster), size=1000, replace=False)
    sample = demo_ace_roster.iloc[indices].copy()
    sample['ace_band'] = pd.cut(sample[ACE],[-.5,.5,1.5,3.5,10.5],labels=['0','1','2–3','4+'])
    exploration_samples[label] = sample
pool_high_band = any(int(s.loc[s[DAYS].notna(),'ace_band'].value_counts().get('4+',0)) < 20
                    for s in exploration_samples.values())
if pool_high_band:
    for sample in exploration_samples.values():
        sample['ace_band'] = pd.cut(sample[ACE],[-.5,.5,1.5,10.5],labels=['0','1','2+'])
print('Band rule:', '0 / 1 / 2+ (prespecified pooling triggered)' if pool_high_band else '0 / 1 / 2–3 / 4+ (no pooling)')
demo_exploration_audit = pd.DataFrame([{'sample':label,'ACE10_eligible_roster_N':len(demo_ace_roster),
    'drawn_n':len(s),'usable_broad_cannabis':int(s[DAYS].notna().sum()),
    'unavailable_broad_cannabis':int(s[DAYS].isna().sum()),
    'usable_COI':int(s[COI_LEVEL].notna().sum()),'unavailable_COI':int(s[COI_LEVEL].isna().sum())}
    for label,s in exploration_samples.items()])
a_availability = demo_exploration_audit.loc[demo_exploration_audit['sample'].eq('A')].iloc[0]
print(f"Sample A: {int(a_availability.drawn_n):,} drawn; cannabis usable={int(a_availability.usable_broad_cannabis):,}, unavailable={int(a_availability.unavailable_broad_cannabis):,}; COI usable={int(a_availability.usable_COI):,}, unavailable={int(a_availability.unavailable_COI):,}.")


In [ ]:
discovery_work = exploration_samples['A'][['ace_band',DAYS]].copy()
discovery_work['is_user'] = discovery_work[DAYS].gt(0)

grouped = discovery_work.groupby('ace_band', observed=False, sort=True)
usable_method = 'count'
user_method = 'sum'
band_counts = grouped.agg(usable_n=(DAYS,usable_method), users=('is_user',user_method))
any_use_pct = 100 * band_counts['users'] / band_counts['usable_n']

# Preserve the complete frozen summary while displaying only Sample A's requested view.
grouped = discovery_work.groupby('ace_band', observed=False, sort=True)
student_discovery_summary = band_counts.assign(any_use_pct=any_use_pct).copy()
student_discovery_summary['sample_band_n'] = grouped.size()
student_discovery_summary['unavailable_n'] = student_discovery_summary['sample_band_n'] - student_discovery_summary['usable_n']
student_discovery_summary['mean_days_all'] = grouped[DAYS].apply(
    lambda v: float(v.dropna().mean()) if v.notna().any() else np.nan)
student_discovery_summary['sparse'] = student_discovery_summary['usable_n'].lt(20)
student_discovery_summary['few_users'] = student_discovery_summary['users'].lt(10)
student_discovery_summary = student_discovery_summary.reset_index().assign(sample='A')
student_discovery_summary['ace_band'] = student_discovery_summary['ace_band'].astype(str)
summary_columns = ['sample','ace_band','sample_band_n','usable_n','unavailable_n','users',
                   'any_use_pct','mean_days_all','sparse','few_users']
student_discovery_summary = student_discovery_summary[summary_columns]
demo_discovery_summary = pd.concat([student_discovery_summary,
    summarize_discovery({'B':exploration_samples['B']}, DAYS)], ignore_index=True)
show(student_discovery_summary[['ace_band','usable_n','users','any_use_pct']])


### Would another sample tell the same story?

Sample B is another prespecified draw of 1,000 people from the same roster, not a later visit. Its band percentages need not repeat Sample A's exactly.


In [ ]:
a = demo_discovery_summary.query("sample == 'A'").drop(columns='sample')
b = demo_discovery_summary.query("sample == 'B'").drop(columns='sample')
demo_exploration_comparison = a[['ace_band','usable_n','users','any_use_pct','mean_days_all']].merge(
    b[['ace_band','usable_n','users','any_use_pct','mean_days_all']],on='ace_band',
    validate='one_to_one',suffixes=('_A','_B'))
exploration_overlap = len(set(exploration_samples['A']['participant_id']) & set(exploration_samples['B']['participant_id']))
changes = '; '.join(f"{r.ace_band}: {r.any_use_pct_A:.1f}% to {r.any_use_pct_B:.1f}%"
                   for r in demo_exploration_comparison.itertuples())
print(f'Sample A to B any-use percentages: {changes}. Shared participants: {exploration_overlap}.')


### Return to the main question

The A–B comparison is a bridge to sampling variability: independent draws from one roster can produce different summaries, and people may appear in both draws but never twice within one sample. We now focus on one estimate—the population mean of the chosen ACE10 or cohesion measure. Its interval does not test differences in cannabis use between ACE10 bands.


## Section 4. Use one sample to estimate a mean

The population is the fictional roster with usable values for the selected measure; `N` is its size. A sample is a smaller group drawn from it, and `n` is the sample size. The supplied sample has n = 75 and is drawn without replacement, so a person cannot appear twice in that sample. The ACE10 demonstration and your chosen measure use separate draws.

### Demo: mean and SD describe different things

The sample mean estimates the population mean. The sample standard deviation (`ddof=1`) describes how much individual scores vary around that sample mean, in the score's original units. It is not uncertainty about the mean.

In [ ]:
from statistics import NormalDist
import platform
from l6_inference_figures import sampling_figure, interval_figure, smoking_figure

N_SMALL, N_LARGE, REPS = 75, 300, 2000
assert analysis_plan['inference_n'] == [N_SMALL,N_LARGE]
assert analysis_plan['classroom_repetitions'] == REPS
assert analysis_plan['confidence'] == .95
demo_values = demo_ace_roster[ACE].to_numpy(dtype=float)
eligible = eligible.sort_values(KEYS).reset_index(drop=True)
values = eligible[field].to_numpy(dtype=float)
assert np.isfinite(demo_values).all() and np.isfinite(values).all()
demo_sample = seed_rng(0,0,N_SMALL).choice(demo_values,size=N_SMALL,replace=False)
sample = seed_rng(meta['role'],0,N_SMALL).choice(values,size=N_SMALL,replace=False)
demo_mean, demo_sd = float(demo_sample.mean()), float(demo_sample.std(ddof=1))
sample_mean, sample_sd = float(sample.mean()), float(sample.std(ddof=1))
demo_score_summary = pd.DataFrame([{'role':'demo','outcome':ACE,'eligible_N':len(demo_values),
    'n':len(demo_sample),'mean':demo_mean,'sample_SD':demo_sd,'units':'ACE categories per person'}])
score_summary = pd.DataFrame([{'role':'your_turn','outcome':field,'eligible_N':len(values),
    'n':len(sample),'mean':sample_mean,'sample_SD':sample_sd,'units':meta['units']}])
show(demo_score_summary); show(score_summary)


### What to notice

Read your mean and SD with their units. An ACE10 mean can be fractional even though each
person has a whole-number count: an average of counts need not be a whole number.
For cohesion, keep the 0–1 score units. Use `score_summary` for **L6-03** to explain what
the mean says about the average and what SD says about differences among people.


## Section 5. One sample to many samples

A new sample of 75 people would usually have a different mean. To study that variability, the supplied experiment draws samples, records each mean, restores the full roster, and repeats 2,000 times at n = 75 and n = 300. A person may appear in different samples.

### Demo: one row is one sample

Each result row records sample size, repetition and sample mean. Repetition alone is not a unique identifier because the same repetition number occurs at both sizes. The invented example stacks records and removes only copied rows; different samples with equal means remain different records.

In [ ]:
# Provided sampling routine; its draws and means use the frozen experiment streams.
def repeat_means(population,n,repetitions,rng):
    population = np.asarray(population,dtype=float)
    assert 2 <= n <= len(population) and repetitions >= 1
    means = []
    for _ in range(repetitions):
        draw = rng.choice(population,size=n,replace=False)
        means.append(float(draw.mean()))
    return np.asarray(means,dtype=float)

demo_repeat_means = repeat_means
demo_sample_means = repeat_means(demo_values,N_SMALL,REPS,seed_rng(0,1,N_SMALL))
demo_large_means = repeat_means(demo_values,N_LARGE,REPS,seed_rng(0,1,N_LARGE))

# Tiny invented result tables: these are not additional roster samples.
toy_result_tables = [
    pd.DataFrame({'n':[2,2], 'repetition':[1,2], 'sample_mean':[1.5,2.0]}),
    pd.DataFrame({'n':[4,4], 'repetition':[1,2], 'sample_mean':[1.75,2.25]})]
toy_repeated = pd.concat(toy_result_tables, ignore_index=True)
show(toy_repeated)
print('Means from the n=2 rows:', toy_repeated.loc[toy_repeated['n'].eq(2), 'sample_mean'].to_numpy())


### Your Turn 2: combine the repeated-sample results

Each row is one repetition at size `n`; `sample_mean` is its result. The third table copies existing records, so it adds no samples. Work in two steps:

1. Choose the fields that identify an experiment record. Compare the same repetition number at different sample sizes, then consider two different repetitions with equal means. Stack the tables and remove only copied records; do not redraw or sort them.
2. Build a Boolean comparison on `n` and use `.loc` to select the requested size, retaining every column and the original row order.

Check the result: it should contain 4,000 genuine rows (2,000 at each size), and each selected table should retain its rows, columns and order. Use these as checks while coding; no written notebook response is required.

**Further learning with Copilot (optional)**

> Use a tiny results table with the same repetition number at two sizes, two equal means from different repetitions, and one copied row. Ask me to predict which rows are duplicates and what the selection should preserve. Give hints about the record key and `.loc`, not my code.


In [ ]:
from l6_structure_checks import check_repeated_table, check_selected_runs

# Provided draws and table columns. This does not draw any new samples after selection.
means_by_n = {n:repeat_means(values,n,REPS,seed_rng(meta['role'],1,n))
              for n in [N_SMALL,N_LARGE]}
repeat_tables = [pd.DataFrame({'n':n, 'repetition':np.arange(1,REPS+1), 'sample_mean':means})
                 for n,means in means_by_n.items()]
# An overlapping export repeats existing records; it makes no new draws.
repeat_tables.append(repeat_tables[0].iloc[:5].copy())
record_key = None  # Fill in the columns that identify one experiment record.
if record_key is None:
    raise ValueError('Your Turn 2: choose the record-identifying columns, then rerun this cell.')
stacked_samples = pd.concat(repeat_tables, ignore_index=True)
repeated_samples = stacked_samples.drop_duplicates(
    subset=record_key, keep='first', ignore_index=True)
check_repeated_table(repeated_samples,means_by_n,REPS)

def means_at_n(results,n):
    size_matches = results['n'].eq(n)
    selected = None  # Use .loc with size_matches; keep every column, value and row order.
    if selected is None:
        raise ValueError('Your Turn 2: complete the row selection inside means_at_n.')
    check_selected_runs(selected,results,n)
    return selected['sample_mean'].to_numpy(dtype=float)

sample_means = means_at_n(repeated_samples,N_SMALL)
large_means = means_at_n(repeated_samples,N_LARGE)
assert sample_means.shape == (REPS,) and large_means.shape == (REPS,)
assert np.isfinite(sample_means).all() and np.isfinite(large_means).all()
print('Clean result table:', len(repeated_samples), 'rows; one unique experiment record per row.')
demo_sampling_figure = sampling_figure(demo_sample,demo_sample_means,demo_large_means,
    units='ACE categories per person',label='Common ACE10 demonstration')
sampling_distribution_figure = sampling_figure(sample,sample_means,large_means,
    units=meta['units'],label='Your '+CHOICE+' workflow')
show(demo_sampling_figure); show(sampling_distribution_figure)


### What to notice

A histogram entry is a sample mean, not one person's score. The distribution of those means shows how an estimate varies across samples; its SD is different from the individual-score SD. More repetitions clarify the simulation but do not add people to any sample. Use `sample_means` for **L6-04**.


## Section 6. Sample size and standard error

The standard error (SE) describes variation in a sample mean across repeated samples. We estimate it from one sample as **sample SD / √n**: SD describes differences among people, while SE describes uncertainty in their average.

Hold SD fixed and increase n from 75 to 300. Four times as many people doubles √n and halves formula-based SE. Separately drawn samples may have different SDs, so their observed ratio need not be exactly two. Compare the formula with the spread of 2,000 repeated means; close does not mean identical.

In [ ]:
se_rows=[]
for role,outcome,sd,means_by_n in [
    ('demo',ACE,demo_sd,{75:demo_sample_means,300:demo_large_means}),
    ('your_turn',field,sample_sd,{75:sample_means,300:large_means})]:
    for n,means in means_by_n.items():
        se_rows.append({'role':role,'outcome':outcome,'n':n,'repetitions':len(means),
            'held_sample_SD':sd,'held_SD_SE':sd/np.sqrt(n),
            'empirical_sampling_SD':means.std(ddof=1),'mean_of_sample_means':means.mean()})
se_comparison=pd.DataFrame(se_rows)
show(se_comparison)
print('Your empirical SD(n=75) / SD(n=300):',sample_means.std(ddof=1)/large_means.std(ddof=1))


### What to notice

In `se_comparison`, `held_SD_SE` uses the same SD at both sizes; `empirical_sampling_SD` is the observed spread of repeated means. Use both for **L6-05**. More people per sample can improve precision; more repetitions only help describe that precision. The simulation draws from the finite roster, while the formula is a large-population approximation.


## Section 7. Put an interval around the estimate

A confidence interval gives a range for the population mean, not for individual scores. For the approximate two-sided 95% interval, calculate **SE = SD / √n**, then **margin = z* × SE**, with normal cutoff z* about 1.96. Subtract the margin from the sample mean for the lower endpoint and add it for the upper endpoint. The calculation uses the sample, not the known population mean; the latter checks the method later.

For **L6-06**, show the numerical substitution and product for `z* × SE`. Use the full stored SE or at least six decimals in the calculation; round only the final margin and endpoints. The table gives enough digits, including for cohesion.

In [ ]:
CONFIDENCE = .95
z95 = NormalDist().inv_cdf((1+CONFIDENCE)/2)
demo_se = demo_sd/np.sqrt(len(demo_sample))
demo_margin = z95*demo_se
demo_interval_result=pd.DataFrame([{'n':len(demo_sample),'mean':demo_mean,'SD':demo_sd,
    'SE':demo_se,'confidence':CONFIDENCE,'z_star':z95,'margin':demo_margin,
    'lower':demo_mean-demo_margin,'upper':demo_mean+demo_margin}])
show(demo_interval_result)


### Apply the calculation to your chosen measure

Trace the sample mean, SD, SE, margin and two endpoints. The calculation does not trim endpoints to the score scale or use the known population mean.


In [ ]:
# Provided: uncertainty of the sample mean, then a two-sided normal-reference interval.
student_se = sample_sd/np.sqrt(len(sample))
student_margin = z95*student_se
student_lower = sample_mean-student_margin
student_upper = sample_mean+student_margin
assert np.isfinite([student_se,student_margin,student_lower,student_upper]).all()
assert student_se > 0 and student_lower < sample_mean < student_upper
interval_result=pd.DataFrame([{'n':len(sample),'mean':sample_mean,'SD':sample_sd,
    'SE':student_se,'confidence':CONFIDENCE,'z_star':z95,'margin':student_margin,
    'lower':student_lower,'upper':student_upper}])
show(interval_result)
print(f'Calculation inputs for L6-06: z* = {z95:.9f}; SE = {student_se:.9f}; mean = {sample_mean:.9f}')


### What does 95% confidence mean?

If the method works as intended, about 95% of intervals from repeated samples include the fixed population mean. This describes the method, not one interval or 95% of individual scores. Use `interval_result` for **L6-06**.


## Section 8. Confidence, coverage and conditions

For the same sample, a higher confidence level uses a larger cutoff and gives a wider interval; it does not add data or repair measurement limits. The table compares 90%, 95% and 99% intervals from that one sample.


In [ ]:
level_comparison=pd.DataFrame([{'n':len(sample),'mean':sample_mean,'SE':student_se,
    'confidence':level,'z_star':NormalDist().inv_cdf((1+level)/2),
    'lower':sample_mean-NormalDist().inv_cdf((1+level)/2)*student_se,
    'upper':sample_mean+NormalDist().inv_cdf((1+level)/2)*student_se}
    for level in [.90,.95,.99]])
level_comparison['width']=level_comparison['upper']-level_comparison['lower']
show(level_comparison)


### Demo: check coverage across all attempts

Coverage is the proportion of interval attempts that include the known population mean. Endpoints equal to the mean count. Keep every attempt in the denominator: an interval with zero or unavailable SE is undefined, not an exact answer and not a hit. The tables include common-demo and selected-measure rows for all 2,000 attempts.


In [ ]:
def interval_simulation(population,n,repetitions,rng,level=.95):
    population=np.asarray(population,dtype=float)
    assert np.isfinite(population).all() and 2 <= n <= len(population) and repetitions >= 1
    assert 0 < level < 1
    z=NormalDist().inv_cdf((1+level)/2)
    rows=[]
    for _ in range(repetitions):
        draw=rng.choice(population,size=n,replace=False)
        mean=float(draw.mean()); sd=float(draw.std(ddof=1)); se=sd/np.sqrt(n)
        valid=bool(np.isfinite(se) and se>0)
        rows.append({'mean':mean,'SD':sd,'SE':se,
            'lower':mean-z*se if valid else np.nan,'upper':mean+z*se if valid else np.nan,'valid':valid})
    return pd.DataFrame(rows)

toy_intervals = pd.DataFrame({'lower':[5.,1.,6.], 'upper':[7.,4.,8.],
                              'valid':[True,True,True]})
toy_target = 5.
toy_intervals['covered'] = (toy_intervals['valid'] & toy_intervals['lower'].le(toy_target)
                            & toy_intervals['upper'].ge(toy_target))
show(toy_intervals)
print('Covered fraction of all invented attempts:', toy_intervals['covered'].mean())

# Published higher-precision estimates, from two frozen 50,000-attempt validation streams.
# This file is evidence for comparison; it does not change classroom draws or settings.
validation_record = json.loads((DATA/'l6_coverage_benchmarks.json').read_text(encoding='utf-8'))
assert validation_record['dataset_version'] == VERSION
validation_rates = pd.DataFrame(validation_record['benchmarks'])
assert not validation_rates.duplicated(['outcome','n']).any()

def compare_coverage_to_validation(summary):
    comparison = summary[['role','outcome','n','repetitions','coverage']].rename(
        columns={'repetitions':'classroom_repetitions','coverage':'classroom_coverage'})
    reference = validation_rates[['outcome','n','replicates','coverage']].rename(
        columns={'replicates':'validation_repetitions','coverage':'validation_coverage'})
    comparison = comparison.merge(reference,on=['outcome','n'],how='left',validate='many_to_one',sort=False)
    assert len(comparison) == len(summary) and comparison.notna().all().all()
    comparison['gap_percentage_points'] = 100*(comparison['classroom_coverage']-comparison['validation_coverage'])
    classroom_variance = comparison['classroom_coverage']*(1-comparison['classroom_coverage'])/comparison['classroom_repetitions']
    validation_variance = comparison['validation_coverage']*(1-comparison['validation_coverage'])/comparison['validation_repetitions']
    comparison['MC_SE_gap_percentage_points'] = 100*np.sqrt(classroom_variance+validation_variance)
    comparison['gap_in_MC_SE'] = comparison['gap_percentage_points']/comparison['MC_SE_gap_percentage_points']
    return comparison

def show_coverage_reference(comparison):
    # Format a display copy; preserve every full-precision value in the exported table.
    assert comparison['classroom_repetitions'].nunique() == 1
    assert comparison['validation_repetitions'].nunique() == 1
    labels = validation_rates.drop_duplicates('outcome').set_index('outcome')['label']
    view = pd.DataFrame({
        'Role':comparison['role'].replace({'demo':'Common demo','your_turn':'Your outcome','conditions_contrast':'Conditions'}),
        'Outcome':comparison['outcome'].map(labels), 'n':comparison['n'],
        'Classroom':comparison['classroom_coverage'].map(lambda x:f'{x:.2%}'),
        'Validation':comparison['validation_coverage'].map(lambda x:f'{x:.3%}'),
        'Gap (pp)':comparison['gap_percentage_points'].map(lambda x:f'{x:+.3f}')})
    print(f"Classroom: {int(comparison['classroom_repetitions'].iloc[0]):,} attempts; "
          f"validation: {int(comparison['validation_repetitions'].iloc[0]):,} attempts. "
          'Gap = classroom minus validation; pp = percentage points. Both rates are estimates.')
    show(view)

def show_reference(title, *items):
    # Extra displays are available on demand; all calculations still run.
    from IPython.display import HTML, display
    from html import escape
    import base64, io
    parts = []
    for item in items:
        if isinstance(item, pd.DataFrame):
            parts.append(item.to_html(index=False))
        else:
            stream = io.BytesIO()
            item.save(stream, format='PNG')
            encoded = base64.b64encode(stream.getvalue()).decode('ascii')
            parts.append('<img alt="' + escape(title) + '" src="data:image/png;base64,' + encoded + '">')
    display(HTML('<h4>' + escape(title) + '</h4>' + ''.join(parts)))


### Your Turn 3: classify intervals and count coverage

Each row is one interval attempt. The supplied `below`, `above` and `undefined` columns classify three outcomes. Fill in the `covered` check: a valid interval covers the target when its lower endpoint is at or below it and its upper endpoint is at or above it. Then calculate the covered proportion using every attempt, including undefined ones.

Before running the cell, mentally classify an interval wholly below the target, one that contains it (including endpoint equality), one wholly above it, and one that is undefined. Every attempt should fit exactly one category. Undefined attempts are not covered, but they stay in the total-attempt denominator. These are self-checks, not an additional written response.

**Further learning with Copilot (optional)**

> Give me a few invented intervals, including endpoint equality and an undefined attempt. Ask me to classify each and identify the denominator before offering a hint. Do not provide my Boolean expression or code.


In [ ]:
from l6_structure_checks import check_interval_flags, check_coverage_rate, check_coverage_function

def classify_intervals(sim,truth):
    classified = sim.copy()
    classified['below'] = sim['valid'] & sim['upper'].lt(truth)
    classified['above'] = sim['valid'] & sim['lower'].gt(truth)
    classified['undefined'] = ~sim['valid']
    covered = None  # One Boolean per attempt: valid and the target is within both endpoints.
    if covered is None:
        raise ValueError('Your Turn 3: complete the covered classification, then rerun this cell.')
    check_interval_flags(sim,truth,covered)
    classified['covered'] = covered
    assert classified[['covered','below','above','undefined']].astype(int).sum(axis=1).eq(1).all()
    return classified[list(sim.columns) + ['covered','below','above','undefined']]

def summarize_coverage(sim,truth):
    classified = classify_intervals(sim,truth)
    rate = None  # Proportion covered, using all rows including undefined attempts.
    if rate is None:
        raise ValueError('Your Turn 3: complete the coverage proportion, then rerun this cell.')
    check_coverage_rate(classified,rate)
    rate = float(rate)
    return {'repetitions':len(classified), 'covered_count':int(classified['covered'].sum()),
        'below_count':int(classified['below'].sum()), 'above_count':int(classified['above'].sum()),
        'undefined_count':int(classified['undefined'].sum()), 'coverage':rate,
        'entirely_below_truth':float(classified['below'].mean()),
        'entirely_above_truth':float(classified['above'].mean()),
        'undefined_rate':float(classified['undefined'].mean()),
        'MC_SE_of_coverage':float(np.sqrt(rate*(1-rate)/len(classified)))}

check_coverage_function(classify_intervals,summarize_coverage)
demo_coverage_runs={n:interval_simulation(demo_values,n,REPS,seed_rng(0,2,n)) for n in [75,300]}
coverage_runs={n:interval_simulation(values,n,REPS,seed_rng(meta['role'],2,n)) for n in [75,300]}
coverage_classifications={n:classify_intervals(sim,float(values.mean())) for n,sim in coverage_runs.items()}
coverage_rows=[]
for role,outcome,population,runs in [('demo',ACE,demo_values,demo_coverage_runs),
                                   ('your_turn',field,values,coverage_runs)]:
    for n,sim in runs.items():
        coverage_rows.append({'role':role,'outcome':outcome,'eligible_N':len(population),'n':n,
            'nominal_confidence':.95,'roster_mean':float(population.mean()),**summarize_coverage(sim,population.mean())})
coverage_summary=pd.DataFrame(coverage_rows)
show(coverage_summary[['role','n','repetitions','covered_count','coverage','MC_SE_of_coverage']])
coverage_reference_comparison = compare_coverage_to_validation(coverage_summary)
show_coverage_reference(coverage_reference_comparison)
interval_plot=interval_figure(coverage_runs[75],float(values.mean()),
    units=meta['units'],label='Your '+CHOICE+' workflow: first 40 attempts')
show(interval_plot)



### Classroom results and the method check

The plot shows your first 40 attempts in their original order: it makes hits and misses visible, but the coverage rate uses all 2,000 attempts. The classroom table uses all 2,000 attempts; the validation comparison uses 100,000 attempts from the same roster and method. Both are estimates. Their gap reflects simulation variability; validation versus 0.95 checks the approximate method. A finite classroom rate alone does not establish systematic over- or undercoverage. Keep the supplied draws even when a result looks unusual.

### A condition failure: smoked-flower days

Smoked-flower days count one product in the same 30-day window. Many people report zero and a few report much higher values, creating a long right tail. The combined display compares the shape and interval coverage at n = 75 and 300; a narrow interval can miss high values and fall below the population mean. Removing nonusers changes the target, and removing undefined attempts hides failures.

In [ ]:
SMOKING='su_y_tlfb__mj__smoke__1mo_ud'
smoking=roster[SMOKING].dropna().to_numpy(dtype=float)
assert len(smoking)==8247
smoke_runs={n:interval_simulation(smoking,n,REPS,seed_rng(3,2,n)) for n in [75,300]}
cannabis_failure_contrast=pd.DataFrame([{'role':'conditions_contrast','outcome':SMOKING,
    'eligible_N':len(smoking),'n':n,'zero_fraction':float((smoking==0).mean()),
    'roster_mean':float(smoking.mean()),**summarize_coverage(sim,smoking.mean())}
    for n,sim in smoke_runs.items()])
shape_rows=[]
for label,population,runs in [(SMOKING,smoking,smoke_runs),(field,values,coverage_runs)]:
    for n,sim in runs.items():
        shape_rows.append({'outcome':label,'n':n,'roster_mean':float(population.mean()),
            'mean_of_sample_means':float(sim['mean'].mean()),'median_of_sample_means':float(sim['mean'].median()),
            'share_below_roster_mean':float(sim['mean'].lt(population.mean()).mean()),
            'empirical_sampling_SD':float(sim['mean'].std(ddof=1))})
sampling_shape_contrast=pd.DataFrame(shape_rows)
smoked_flower_figure=smoking_figure(smoking,smoke_runs[75],smoke_runs[300],float(smoking.mean()))
smoking_reference_comparison = compare_coverage_to_validation(cannabis_failure_contrast)
show_reference("Smoked-flower shape and coverage", smoked_flower_figure,
               sampling_shape_contrast.query("outcome == @SMOKING"),
               cannabis_failure_contrast, smoking_reference_comparison)


### What to notice

Validation shows smoked-flower coverage below 95% at both sizes, though it improves at n = 300. More data may help, but shape still matters. For **L6-08**, use the smoked-flower display and validation comparison. For **L6-07**, use the complete coverage tables to report classroom coverage, Monte Carlo SE and classroom-versus-validation results.


## Section 9. Answer the main question

The selected sample mean estimates the average among usable values in the fictional roster; its interval summarizes sampling uncertainty. Report the estimate, units and interval in Canvas item **L6-09**, along with one Sample A any-use finding and its usable-report denominator. Add one limitation that a narrower interval cannot fix, chosen from linkage provenance, location privacy or ecological inference.

Use the plan, join checks, Sample A table and `results_summary` as evidence. Canvas is the only required written reflection; no written notebook response or notebook upload is required. After completing the three Your Turns, restart the kernel and run your notebook from top to bottom. Leave the optional bounds extension off.


In [ ]:
results_summary=pd.DataFrame([{'outcome':field,'eligible_N':len(values),'n':len(sample),
    'mean':sample_mean,'SD':sample_sd,'SE':student_se,'confidence':.95,
    'lower':student_lower,'upper':student_upper,'units':meta['units']}])
show(results_summary)
result_id=f'L6-{VERSION}-{NOTEBOOK_VERSION}-{CHOICE}-n75-300-root6060'
results_record={'result_id':result_id,'plan':analysis_plan,
    'runtime':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__},
    'source_record':source_record.to_dict('records'),
    'linkage_audit':linkage_audit.to_dict('records'),'population_audit':population_audit.to_dict('records'),
    'demo_exploration_audit':demo_exploration_audit.to_dict('records'),
    'demo_discovery_summary':demo_discovery_summary.to_dict('records'),
    'score_summary':score_summary.to_dict('records'),'se_comparison':se_comparison.to_dict('records'),
    'interval_result':interval_result.to_dict('records'),'level_comparison':level_comparison.to_dict('records'),
    'coverage_summary':coverage_summary.to_dict('records'),
    'coverage_reference_comparison':coverage_reference_comparison.to_dict('records'),
    'smoking_reference_comparison':smoking_reference_comparison.to_dict('records'),
    'sampling_shape_contrast':sampling_shape_contrast.to_dict('records'),
    'cannabis_failure_contrast':cannabis_failure_contrast.to_dict('records'),
    'results_summary':results_summary.to_dict('records')}
results_path=Path(f'lab6_{CHOICE}_results.json')
results_path.write_text(json.dumps(results_record,indent=2,ensure_ascii=False),encoding='utf-8')
print('Result ID:',result_id)
print('Saved your local reference record:',results_path.name)


### Prepare the Canvas reflection

Use the notebook evidence for Canvas items **L6-01–L6-10**. The first items cover the estimand and join; the later items cover sample summaries, repeated means, interval calculation, coverage, conditions and synthesis. Item 10 is the AI-use attestation. Complete and submit the reflection in Canvas; do not upload this notebook.

**The required notebook work ends here.**


## Optional extension. Describe an upper or lower limit

You have completed the core lab. This extension explores a different kind of question and
is not needed for the required mean estimate or results record. Leave `RUN_OPTIONAL_BOUND`
set to `False` to skip its calculations. To explore it, first complete the core lab, then
change the switch to `True` and run the extension cells.

Our two-sided interval gives both a lower and an upper endpoint. Sometimes a question asks
for uncertainty in only one direction: an **upper bound** gives an upper limit for the
population mean, while a **lower bound** gives a lower limit. Neither describes a limit
on individual people's scores.

### Demo: an upper bound for the ACE10 mean

For a two-sided 95% interval, the normal-curve calculation leaves 2.5% in each tail.
A one-sided 95% bound places the full 5% on one side. Its cutoff is about **1.645**, rather
than 1.96. An upper bound adds this cutoff times SE to the sample mean; a lower bound subtracts it.

The extension uses a fixed rule: **upper for ACE10, lower for cohesion**. This rule is part
of the exercise design and does not depend on observed results. Choosing whichever side
looks favorable afterward would change the question in response to the answer. Like our
interval, this is an approximate method, not a guarantee of exact 95% coverage.

**Optional reading:** §4.2.3. Neither this reading nor the extension is assessed. One- versus two-sided reasoning returns conceptually in Module 7.


In [ ]:
RUN_OPTIONAL_BOUND = False  # Optional: change to True after completing the core lab.
OPTIONAL_BOUND_DIRECTION = {'adversity':'upper', 'cohesion':'lower'}


In [ ]:
if RUN_OPTIONAL_BOUND:
    z_one=NormalDist().inv_cdf(.95)
    demo_one_sided_bound=pd.DataFrame([{'n':len(demo_sample),'mean':demo_mean,'SE':demo_se,
        'direction':'upper','confidence':.95,'z_star':z_one,'bound':demo_mean+z_one*demo_se,
        'corresponding_two_sided_endpoint':float(demo_interval_result.iloc[0]['upper'])}])
    show(demo_one_sided_bound)
else:
    print('Optional bound calculations skipped; the core lab is complete.')


### Apply the extension's fixed direction

If you turned on the extension, the next cell uses its fixed direction for your selected
measure. Follow whether it adds or subtracts the margin, then compare the bound with the
corresponding endpoint of your two-sided interval. Both use the same sample mean and SE.
The optional tables do not change the core results record you already saved.


In [ ]:
if RUN_OPTIONAL_BOUND:
    direction=OPTIONAL_BOUND_DIRECTION[CHOICE]
    assert direction in ('upper','lower')
    # The extension rule depends on the measure, never on the observed result.
    student_bound = sample_mean + z_one*student_se if direction=='upper' else sample_mean - z_one*student_se
    assert np.isfinite(student_bound)
    assert (sample_mean < student_bound < student_upper if direction=='upper' else student_lower < student_bound < sample_mean)
    one_sided_bound=pd.DataFrame([{'n':len(sample),'mean':sample_mean,'SE':student_se,
        'direction':direction,'confidence':.95,'z_star':z_one,'bound':student_bound,
        'corresponding_two_sided_endpoint':student_upper if direction=='upper' else student_lower}])
    show(one_sided_bound)


### What to notice — optional

State whether the result gives an upper or lower limit for the population mean, using your
measure's units. Why is its endpoint closer to the sample mean than the corresponding
two-sided 95% endpoint? Relate the difference to the two cutoffs. There is no additional
required notebook response or Canvas bound calculation for this extension.
